In [ ]:
# Data Quality Assessment

## Objective

Assess missing values, duplicate records, data types,
consistency and suspicious values in NSL-KDD and UNSW-NB15.

All checks are performed on the original train and test
partitions without modifying the source datasets.

In [1]:
import pandas as pd
from pathlib import Path

current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_dir = current_dir.parent
else:
    project_dir = current_dir

print("Project directory:", project_dir)

Project directory: C:\Users\fatim\Documents\network-intrusion-eda


In [2]:
nsl_columns = [
    "duration", "protocol_type", "service", "flag",
    "src_bytes", "dst_bytes", "land", "wrong_fragment",
    "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted",
    "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate",
    "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate",
    "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "attack_type", "difficulty_level"
]

nsl_train = pd.read_csv(
    project_dir / "data/raw/nsl_kdd/KDDTrain+.txt",
    header=None,
    names=nsl_columns
)

nsl_test = pd.read_csv(
    project_dir / "data/raw/nsl_kdd/KDDTest+.txt",
    header=None,
    names=nsl_columns
)

unsw_train = pd.read_csv(
    project_dir / "data/raw/unsw_nb15/UNSW_NB15_training-set.csv"
)

unsw_test = pd.read_csv(
    project_dir / "data/raw/unsw_nb15/UNSW_NB15_testing-set.csv"
)

datasets = {
    "NSL-KDD Train": nsl_train,
    "NSL-KDD Test": nsl_test,
    "UNSW-NB15 Train": unsw_train,
    "UNSW-NB15 Test": unsw_test
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

NSL-KDD Train: (125973, 43)
NSL-KDD Test: (22544, 43)
UNSW-NB15 Train: (175341, 45)
UNSW-NB15 Test: (82332, 45)


In [3]:
missing_summary = []

for name, df in datasets.items():

    missing_cells = int(df.isna().sum().sum())

    total_cells = df.shape[0] * df.shape[1]

    missing_percentage = (
        missing_cells / total_cells * 100
        if total_cells > 0 else 0
    )

    missing_summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Cells": missing_cells,
        "Missing %": round(missing_percentage, 4)
    })

missing_df = pd.DataFrame(missing_summary)

display(missing_df)

,Dataset,Rows,Columns,Missing Cells,Missing %
0,NSL-KDD Train,125973,43,0,0.0
1,NSL-KDD Test,22544,43,0,0.0
2,UNSW-NB15 Train,175341,45,0,0.0
3,UNSW-NB15 Test,82332,45,0,0.0


In [4]:
blank_summary = []

for name, df in datasets.items():

    text_columns = df.select_dtypes(
        include=["object", "str", "category"]
    )

    blank_count = 0

    for column in text_columns.columns:

        blank_count += int(
            text_columns[column]
            .astype("string")
            .str.strip()
            .eq("")
            .sum()
        )

    blank_summary.append({
        "Dataset": name,
        "Blank or Whitespace Cells": blank_count
    })

blank_df = pd.DataFrame(blank_summary)

display(blank_df)

,Dataset,Blank or Whitespace Cells
0,NSL-KDD Train,0
1,NSL-KDD Test,0
2,UNSW-NB15 Train,0
3,UNSW-NB15 Test,0


In [ ]:
## Duplicate-Record Assessment

In [5]:
duplicate_results = []

for name, df in datasets.items():

    # Exclude the record identifier from UNSW-NB15
    if name.startswith("UNSW-NB15"):
        comparison_df = df.drop(columns=["id"])
    else:
        comparison_df = df

    duplicate_count = int(
        comparison_df.duplicated().sum()
    )

    duplicate_percentage = (
        duplicate_count / len(df) * 100
    )

    duplicate_results.append({
        "Dataset": name,
        "Total Records": len(df),
        "Duplicate Records": duplicate_count,
        "Duplicate %": round(duplicate_percentage, 4)
    })

duplicate_df = pd.DataFrame(duplicate_results)

display(duplicate_df)

,Dataset,Total Records,Duplicate Records,Duplicate %
0,NSL-KDD Train,125973,0,0.0000
1,NSL-KDD Test,22544,0,0.0000
2,UNSW-NB15 Train,175341,67601,38.5540
3,UNSW-NB15 Test,82332,26387,32.0495


In [6]:
feature_duplicate_results = []

for name, df in datasets.items():

    if name.startswith("NSL-KDD"):

        excluded_columns = [
            "attack_type",
            "difficulty_level"
        ]

    else:

        excluded_columns = [
            "id",
            "attack_cat",
            "label"
        ]

    feature_columns = [
        col for col in df.columns
        if col not in excluded_columns
    ]

    feature_duplicates = int(
        df.duplicated(
            subset=feature_columns
        ).sum()
    )

    feature_duplicate_results.append({
        "Dataset": name,
        "Feature Columns": len(feature_columns),
        "Repeated Feature Rows": feature_duplicates,
        "Repeated Feature %": round(
            feature_duplicates / len(df) * 100, 4
        )
    })

feature_duplicate_df = pd.DataFrame(
    feature_duplicate_results
)

display(feature_duplicate_df)

,Dataset,Feature Columns,Repeated Feature Rows,Repeated Feature %
0,NSL-KDD Train,41,16,0.0127
1,NSL-KDD Test,41,57,0.2528
2,UNSW-NB15 Train,42,74301,42.3751
3,UNSW-NB15 Test,42,28386,34.4775


In [7]:
print("Dataset dimensions after duplicate checks:\n")

for name, df in datasets.items():
    print(name, df.shape)

Dataset dimensions after duplicate checks:

NSL-KDD Train (125973, 43)
NSL-KDD Test (22544, 43)
UNSW-NB15 Train (175341, 45)
UNSW-NB15 Test (82332, 45)


In [ ]:
## Duplicate Distribution by Traffic Class

In [8]:
duplicate_class_results = []

for name, df in datasets.items():

    if name.startswith("NSL-KDD"):

        comparison_columns = df.columns.tolist()

        traffic_class = df["attack_type"].apply(
            lambda x: "Normal" if x == "normal" else "Attack"
        )

    else:

        comparison_columns = [
            col for col in df.columns if col != "id"
        ]

        traffic_class = df["label"].map({
            0: "Normal",
            1: "Attack"
        })

    duplicate_mask = df.duplicated(
        subset=comparison_columns,
        keep="first"
    )

    for category in ["Normal", "Attack"]:

        class_mask = traffic_class == category

        total_records = int(class_mask.sum())

        duplicate_records = int(
            (duplicate_mask & class_mask).sum()
        )

        duplicate_percentage = (
            duplicate_records / total_records * 100
            if total_records > 0 else 0
        )

        duplicate_class_results.append({
            "Dataset": name,
            "Traffic Class": category,
            "Total Records": total_records,
            "Duplicate Records": duplicate_records,
            "Duplicate %": round(
                duplicate_percentage, 2
            )
        })

duplicate_class_df = pd.DataFrame(
    duplicate_class_results
)

display(duplicate_class_df)

,Dataset,Traffic Class,Total Records,Duplicate Records,Duplicate %
0,NSL-KDD Train,Normal,67343,0,0.00
1,NSL-KDD Train,Attack,58630,0,0.00
2,NSL-KDD Test,Normal,9711,0,0.00
3,NSL-KDD Test,Attack,12833,0,0.00
4,UNSW-NB15 Train,Normal,56000,4110,7.34
5,UNSW-NB15 Train,Attack,119341,63491,53.20
6,UNSW-NB15 Test,Normal,37000,2794,7.55
7,UNSW-NB15 Test,Attack,45332,23593,52.04


In [9]:
conflict_results = []

for name, df in datasets.items():

    if name.startswith("NSL-KDD"):

        feature_columns = [
            col for col in df.columns
            if col not in [
                "attack_type",
                "difficulty_level"
            ]
        ]

        label_column = "attack_type"

    else:

        feature_columns = [
            col for col in df.columns
            if col not in [
                "id",
                "attack_cat",
                "label"
            ]
        ]

        label_column = "label"

    label_counts = (
        df.groupby(
            feature_columns,
            dropna=False
        )[label_column]
        .nunique()
    )

    conflicting_patterns = int(
        (label_counts > 1).sum()
    )

    conflict_results.append({
        "Dataset": name,
        "Conflicting Feature Patterns": conflicting_patterns
    })

conflict_df = pd.DataFrame(conflict_results)

display(conflict_df)

,Dataset,Conflicting Feature Patterns
0,NSL-KDD Train,15
1,NSL-KDD Test,57
2,UNSW-NB15 Train,229
3,UNSW-NB15 Test,6


In [10]:
print("Original duplicate counts:")

display(
    duplicate_df[
        ["Dataset", "Duplicate Records"]
    ]
)

print("\nDuplicates grouped by traffic class:")

display(
    duplicate_class_df.groupby(
        "Dataset",
        as_index=False
    )["Duplicate Records"].sum()
)

Original duplicate counts:


,Dataset,Duplicate Records
0,NSL-KDD Train,0
1,NSL-KDD Test,0
2,UNSW-NB15 Train,67601
3,UNSW-NB15 Test,26387



Duplicates grouped by traffic class:


,Dataset,Duplicate Records
0,NSL-KDD Test,0
1,NSL-KDD Train,0
2,UNSW-NB15 Test,26387
3,UNSW-NB15 Train,67601


In [ ]:
## Feature Validity and Suspicious Values

In [11]:
validity_checks = [
    ("NSL-KDD Train", nsl_train, ["duration", "src_bytes", "dst_bytes"]),
    ("NSL-KDD Test", nsl_test, ["duration", "src_bytes", "dst_bytes"]),
    ("UNSW-NB15 Train", unsw_train, ["dur", "sbytes", "dbytes"]),
    ("UNSW-NB15 Test", unsw_test, ["dur", "sbytes", "dbytes"])
]

negative_results = []

for name, df, columns in validity_checks:

    for column in columns:

        negative_count = int((df[column] < 0).sum())

        negative_results.append({
            "Dataset": name,
            "Feature": column,
            "Minimum": df[column].min(),
            "Maximum": df[column].max(),
            "Negative Values": negative_count
        })

negative_df = pd.DataFrame(negative_results)

display(negative_df)

,Dataset,Feature,Minimum,Maximum,Negative Values
0,NSL-KDD Train,duration,0.0,4.290800e+04,0
1,NSL-KDD Train,src_bytes,0.0,1.379964e+09,0
2,NSL-KDD Train,dst_bytes,0.0,1.309937e+09,0
3,NSL-KDD Test,duration,0.0,5.771500e+04,0
4,NSL-KDD Test,src_bytes,0.0,6.282565e+07,0
5,NSL-KDD Test,dst_bytes,0.0,1.345927e+06,0
6,UNSW-NB15 Train,dur,0.0,5.999999e+01,0
7,UNSW-NB15 Train,sbytes,28.0,1.296523e+07,0
8,UNSW-NB15 Train,dbytes,0.0,1.465555e+07,0
9,UNSW-NB15 Test,dur,0.0,5.999999e+01,0


In [12]:
rate_columns = [
    "serror_rate",
    "srv_serror_rate",
    "rerror_rate",
    "srv_rerror_rate",
    "same_srv_rate",
    "diff_srv_rate",
    "srv_diff_host_rate",
    "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate",
    "dst_host_srv_serror_rate",
    "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate"
]

rate_results = []

for name, df in [
    ("NSL-KDD Train", nsl_train),
    ("NSL-KDD Test", nsl_test)
]:

    for column in rate_columns:

        outside_range = int(
            ((df[column] < 0) | (df[column] > 1)).sum()
        )

        rate_results.append({
            "Dataset": name,
            "Feature": column,
            "Minimum": df[column].min(),
            "Maximum": df[column].max(),
            "Outside 0–1": outside_range
        })

rate_df = pd.DataFrame(rate_results)

display(rate_df)

,Dataset,Feature,Minimum,Maximum,Outside 0–1
0,NSL-KDD Train,serror_rate,0.0,1.0,0
1,NSL-KDD Train,srv_serror_rate,0.0,1.0,0
2,NSL-KDD Train,rerror_rate,0.0,1.0,0
3,NSL-KDD Train,srv_rerror_rate,0.0,1.0,0
4,NSL-KDD Train,same_srv_rate,0.0,1.0,0
5,NSL-KDD Train,diff_srv_rate,0.0,1.0,0
6,NSL-KDD Train,srv_diff_host_rate,0.0,1.0,0
7,NSL-KDD Train,dst_host_same_srv_rate,0.0,1.0,0
8,NSL-KDD Train,dst_host_diff_srv_rate,0.0,1.0,0
9,NSL-KDD Train,dst_host_same_src_port_rate,0.0,1.0,0


In [13]:
binary_results = []

for name, df in [
    ("UNSW-NB15 Train", unsw_train),
    ("UNSW-NB15 Test", unsw_test)
]:

    invalid_count = int(
        (~df["label"].isin([0, 1])).sum()
    )

    binary_results.append({
        "Dataset": name,
        "Unique Labels": sorted(df["label"].unique().tolist()),
        "Invalid Binary Labels": invalid_count
    })

binary_df = pd.DataFrame(binary_results)

display(binary_df)

,Dataset,Unique Labels,Invalid Binary Labels
0,UNSW-NB15 Train,"[0, 1]",0
1,UNSW-NB15 Test,"[0, 1]",0


In [14]:
label_consistency_results = []

for name, df in [
    ("UNSW-NB15 Train", unsw_train),
    ("UNSW-NB15 Test", unsw_test)
]:

    inconsistent_mask = (
        ((df["label"] == 0) & (df["attack_cat"] != "Normal"))
        |
        ((df["label"] == 1) & (df["attack_cat"] == "Normal"))
    )

    inconsistent_count = int(inconsistent_mask.sum())

    label_consistency_results.append({
        "Dataset": name,
        "Inconsistent Labels": inconsistent_count
    })

label_consistency_df = pd.DataFrame(
    label_consistency_results
)

display(label_consistency_df)

,Dataset,Inconsistent Labels
0,UNSW-NB15 Train,0
1,UNSW-NB15 Test,0


In [ ]:
## Train–Test Data Quality Comparison

In [15]:
quality_summary = (
    missing_df
    .merge(
        duplicate_df[
            ["Dataset", "Duplicate Records", "Duplicate %"]
        ],
        on="Dataset"
    )
    .merge(
        feature_duplicate_df[
            ["Dataset", "Repeated Feature Rows", "Repeated Feature %"]
        ],
        on="Dataset"
    )
    .merge(
        conflict_df,
        on="Dataset"
    )
)

display(quality_summary)

,Dataset,Rows,Columns,Missing Cells,Missing %,Duplicate Records,Duplicate %,Repeated Feature Rows,Repeated Feature %,Conflicting Feature Patterns
0,NSL-KDD Train,125973,43,0,0.0,0,0.0000,16,0.0127,15
1,NSL-KDD Test,22544,43,0,0.0,0,0.0000,57,0.2528,57
2,UNSW-NB15 Train,175341,45,0,0.0,67601,38.5540,74301,42.3751,229
3,UNSW-NB15 Test,82332,45,0,0.0,26387,32.0495,28386,34.4775,6


In [16]:
class_distribution = []

for name, df in datasets.items():

    if name.startswith("NSL-KDD"):
        attack_count = int(
            (df["attack_type"] != "normal").sum()
        )
    else:
        attack_count = int(
            (df["label"] == 1).sum()
        )

    normal_count = len(df) - attack_count

    class_distribution.append({
        "Dataset": name,
        "Normal Records": normal_count,
        "Attack Records": attack_count,
        "Attack %": round(
            attack_count / len(df) * 100, 2
        )
    })

class_distribution_df = pd.DataFrame(
    class_distribution
)

quality_summary = quality_summary.merge(
    class_distribution_df,
    on="Dataset"
)

display(
    quality_summary[
        [
            "Dataset",
            "Rows",
            "Missing Cells",
            "Duplicate %",
            "Repeated Feature %",
            "Conflicting Feature Patterns",
            "Attack %"
        ]
    ]
)

,Dataset,Rows,Missing Cells,Duplicate %,Repeated Feature %,Conflicting Feature Patterns,Attack %
0,NSL-KDD Train,125973,0,0.0000,0.0127,15,46.54
1,NSL-KDD Test,22544,0,0.0000,0.2528,57,56.92
2,UNSW-NB15 Train,175341,0,38.5540,42.3751,229,68.06
3,UNSW-NB15 Test,82332,0,32.0495,34.4775,6,55.06


In [17]:
output_path = (
    project_dir
    / "reports"
    / "data_quality_summary.csv"
)

quality_summary.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("File exists:", output_path.exists())

Saved: C:\Users\fatim\Documents\network-intrusion-eda\reports\data_quality_summary.csv
File exists: True


In [18]:
print("Quality summary dimensions:")
print(quality_summary.shape)

print("\nDatasets included:")
print(quality_summary["Dataset"].tolist())

print("\nMissing values in summary:")
print(quality_summary.isna().sum().sum())

Quality summary dimensions:
(4, 13)

Datasets included:
['NSL-KDD Train', 'NSL-KDD Test', 'UNSW-NB15 Train', 'UNSW-NB15 Test']

Missing values in summary:
0
